In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms  # 테스트용 데이터셋 추가

# 장치 설정 (Mac M1/M2 환경 최적화)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# --- 임시 데이터로더 생성 (실행 가능하도록 추가) ---
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
# ---------------------------------------------

# Generator: 노이즈 -> 이미지
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh() # MNIST 출력을 -1 ~ 1 사이로 맞춤
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)
    
# Discriminator: 이미지 -> 확률
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2), 
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x.view(-1, 784))
    
# 초기화
G = Generator().to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

for epoch in range(100):
    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        B = real_imgs.size(0)

        # ==========================================
        # 1. Discriminator 학습: BCE 최소화
        # ==========================================
        opt_D.zero_grad()

        # 진짜 이미지 -> D -> 1에 가깝게
        real_loss = criterion(D(real_imgs), torch.ones(B, 1).to(device))

        # 가짜 이미지 -> D -> 0에 가깝게
        z = torch.randn(B, 100).to(device)
        fake_imgs = G(z).detach()  # G의 기울기 차단
        fake_loss = criterion(D(fake_imgs), torch.zeros(B, 1).to(device))

        loss_D = real_loss + fake_loss
        loss_D.backward()
        opt_D.step()

        # ==========================================
        # 2. Generator 학습: D를 속이도록 가중치 유도
        # ==========================================
        opt_G.zero_grad()

        # 가짜 이미지를 진짜(1)로 속이도록
        z = torch.randn(B, 100).to(device)
        gen_imgs = G(z)
        loss_G = criterion(D(gen_imgs), torch.ones(B, 1).to(device))

        loss_G.backward()
        opt_G.step()

    print(f"Epoch {epoch} : D = {loss_D.item():.3f}, G = {loss_G.item():.3f}")

Failed to download (trying next):
HTTP Error 404: Not Found

Failed to download (trying next):
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>



RuntimeError: Error downloading train-images-idx3-ubyte.gz